In [69]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from torch.nn import TransformerEncoder, TransformerEncoderLayer

In [57]:
num_files=15
file_prefix="/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_"
all_dfs = []

for i in range(1, num_files+1):
    df = pd.read_pickle(f"{file_prefix}{i}.pkl.zip", compression="zip")
    df = df[['x', 'y', 'z', 'time', 'latent_representation']]
    df['latent_representation'] = df['latent_representation'].apply(lambda x: x[0])
    all_dfs.append(df)

# Concatenate all dataframes
df_combined = pd.concat(all_dfs, ignore_index=True)
print(df_combined.head())
# Pivot table to get v as a matrix with shape (num_samples, 1200, 48)
df_pivot = df_combined.pivot_table(index=['x','y','z'], columns='time', values='latent_representation', aggfunc='first')


     x   y   z  time                              latent_representation
0 -117  79 -25     1  [0.50288206, -0.6009416, -0.16422167, 0.923491...
1 -113  79 -25     1  [-0.5462359, -0.47092205, -0.1574297, -0.10681...
2 -109  79 -25     1  [-0.3031874, -0.6288186, -0.5930966, -0.788726...
3 -105  79 -25     1  [0.36541444, 0.4556977, -0.7264514, -0.6433497...
4 -101  79 -25     1  [-0.9766785, 0.768204, -0.06921359, -0.8455191...


In [58]:
df_pivot.head()

time                                                         1   \
x    y   z                                                        
-117 -76 -25  [0.60152996, 0.98103255, -0.008870157, -0.2454...   
         -21  [0.096365936, 0.54300946, 0.8387079, 0.4684101...   
         -17  [0.6943935, 0.6303967, -0.8195155, 0.9442892, ...   
         -13  [0.56154877, 0.7658759, -0.31268513, 0.9745437...   
         -9   [0.9743955, -0.93269074, 0.027911449, 0.220510...   

time                                                         2   \
x    y   z                                                        
-117 -76 -25  [0.5743986, 0.7089401, 0.5678579, -0.93600154,...   
         -21  [-0.79866904, -0.12209187, 0.91154915, 0.76043...   
         -17  [0.9876632, -0.61406004, -0.3469003, 0.4697731...   
         -13  [-0.7751982, -0.325407, 0.4743654, -0.13019444...   
         -9   [-0.3710597, -0.19291241, 0.8153351, -0.286354...   

time                                                         3   \
x    y   z                                                        
-117 -76 -25  [-0.8775012, 0.77081597, 0.34914115, 0.9433937...   
         -21  [0.22371271, -0.7035653, -0.24191181, -0.04110...   
         -17  [-0.94120944, 0.9091235, 0.63393474, 0.7522126...   
         -13  [0.31826624, 0.65888816, 0.98721373, -0.370199...   
         -9   [-0.9283217, 0.25671372, 0.90663964, 0.91436, ...   

time                                                         4   \
x    y   z                                                        
-117 -76 -25  [-0.21666437, 0.276739, 0.67670155, -0.8597325...   
         -21  [-0.12166012, -0.90468264, -0.703625, -0.35087...   
         -17  [0.9533979, 0.374611, 0.08358036, 0.45856717, ...   
         -13  [-0.73936015, 0.13778922, -0.73885727, -0.4434...   
         -9   [-0.44492516, -0.2988802, -0.15240346, -0.3305...   

time                                                         5   \
x    y   z                                                        
-117 -76 -25  [0.36412022, -0.76997614, 0.266663, -0.4527933...   
         -21  [-0.6756259, 0.20125556, 0.41114074, -0.192610...   
         -17  [0.103173874, -0.9596432, -0.21567442, 0.48207...   
         -13  [0.95708394, 0.9988378, -0.84009784, -0.483040...   
         -9   [-0.5891063, -0.2228696, -0.18154225, -0.36551...   

time                                                         6   \
x    y   z                                                        
-117 -76 -25  [-0.3553231, -0.37004066, 0.40704593, -0.85804...   
         -21  [-0.2619261, -0.9286362, 0.87315434, 0.1708212...   
         -17  [0.8844934, -0.4611117, -0.35537854, -0.904041...   
         -13  [-0.43986663, -0.8859522, -0.60281193, -0.9730...   
         -9   [0.70254093, 0.91327983, 0.54900235, 0.8643656...   

time                                                         7   \
x    y   z                                                        
-117 -76 -25  [0.4651682, 0.32191125, -0.8944155, 0.7075796,...   
         -21  [0.12846749, -0.70874083, -0.09848327, 0.54393...   
         -17  [0.19190139, -0.2850105, 0.09330799, 0.2111620...   
         -13  [0.628395, 0.42571187, -0.22475213, -0.5347901...   
         -9   [-0.8378765, -0.8759353, 0.020267947, -0.99677...   

time                                                         8   \
x    y   z                                                        
-117 -76 -25  [-0.7426088, 0.7396835, -0.4338635, 0.6320076,...   
         -21  [-0.9192027, 0.39118025, 0.6679161, -0.7074012...   
         -17  [-0.55416745, 0.42543814, 0.32234558, -0.66634...   
         -13  [0.8604549, -0.80334365, -0.19985121, 0.601780...   
         -9   [-0.43921062, 0.9817179, 0.47819474, 0.8692322...   

time                                                         9   \
x    y   z                                                        
-117 -76 -25  [-0.16479653, 0.6672011, 0.44587672, -0.632495...   
         -21  [0.75533044, -0.95989746,

In [124]:
df_pivot.shape

(33600, 15)

In [68]:
df_pivot.index

MultiIndex([(-117, -76, -25),
            (-117, -76, -21),
            (-117, -76, -17),
            (-117, -76, -13),
            (-117, -76,  -9),
            (-117, -76,  -5),
            (-117, -76,  -1),
            (-117, -76,   3),
            (-117, -76,   7),
            (-117, -76,  11),
            ...
            ( 117,  79,  -9),
            ( 117,  79,  -5),
            ( 117,  79,  -1),
            ( 117,  79,   3),
            ( 117,  79,   7),
            ( 117,  79,  11),
            ( 117,  79,  14),
            ( 117,  79,  18),
            ( 117,  79,  22),
            ( 117,  79,  26)],
           names=['x', 'y', 'z'], length=33600)

In [87]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df_pivot):
        self.df = df_pivot

    def __len__(self):
        return len(self.df) * (self.df.columns.size - 4)  # sliding window of 4

    def __getitem__(self, idx):
        row_idx = idx // (self.df.columns.size - 4)
        col_idx = idx % (self.df.columns.size - 4)

        # Convert the nested arrays (objects) into a proper tensor format
        src_values = [self.df.iloc[row_idx, col_idx + i] for i in range(4)]
        x = torch.tensor(src_values, dtype=torch.float32)

        y_values = self.df.iloc[row_idx, col_idx + 4]
        y = torch.tensor(y_values, dtype=torch.float32)

        return x, y


In [79]:
class TransformerSeq2Seq(nn.Module):
    def __init__(self, d_model, nhead, num_layers, dim_feedforward):
        super(TransformerSeq2Seq, self).__init__()

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_layers,
            num_decoder_layers=num_layers,
            dim_feedforward=dim_feedforward
        )
        self.fc = nn.Linear(d_model, 48)  # To match target size

    def forward(self, src, tgt):
        output = self.transformer(src, tgt)
        return self.fc(output)


In [85]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, (src, tgt) in enumerate(loader):
        src, tgt = src.to(device), tgt.to(device)

        # Repeat the target to match the source's sequence length
        tgt_repeated = tgt.repeat(src.shape[1], 1, 1).permute(1, 0, 2)

        optimizer.zero_grad()
        output = model(src, tgt_repeated[:-1])  # use all but the last timestep of tgt as input
        loss = criterion(output[-1:], tgt)  # only compute loss for the last timestep
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)


In [81]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [82]:
dataset = TimeSeriesDataset(df_pivot)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [83]:
# Parameters
d_model = 48
nhead = 4
num_layers = 2
dim_feedforward = 512
epochs = 10
learning_rate = 0.001


# Load data


# Model, Loss, Optimizer
model = TransformerSeq2Seq(d_model, nhead, num_layers, dim_feedforward).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [86]:


# Training loop
for epoch in range(epochs):
    loss = train(model, loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1}/{epochs} Loss: {loss:.4f}")

/mnt/d/sources/Interpreter/pytorch/lib/python3.8/site-packages/torch/nn/modules/loss.py:536: UserWarning: Using a target size (torch.Size([32, 48])) that is different to the input size (torch.Size([1, 4, 48])). This will likely lead to incorrect results due to broadcasting. Please ensure they have the same size.
  return F.mse_loss(input, target, reduction=self.reduction)


RuntimeError: The size of tensor a (4) must match the size of tensor b (32) at non-singleton dimension 1

In [99]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df_pivot):
        self.df = df_pivot

    def __len__(self):
        return len(self.df) * (self.df.columns.size - 4)  # sliding window of 4

    def __getitem__(self, idx):
        row_idx = idx // (self.df.columns.size - 4)
        col_idx = idx % (self.df.columns.size - 4)

        src_values = [self.df.iloc[row_idx, col_idx + i] for i in range(4)]
        x = torch.tensor(src_values, dtype=torch.float32)

        y_values = self.df.iloc[row_idx, col_idx + 4]
        y = torch.tensor(y_values, dtype=torch.float32)

        return x, y


In [95]:
import torch.nn as nn
from torch.nn import TransformerEncoder, TransformerEncoderLayer

class Seq2PointTransformer(nn.Module):
    def __init__(self, d_model, nhead, num_encoder_layers, dim_feedforward):
        super(Seq2PointTransformer, self).__init__()
        self.model_type = 'Transformer'

        # Encoder part
        self.encoder_layer = TransformerEncoderLayer(d_model, nhead, dim_feedforward)
        self.transformer_encoder = TransformerEncoder(self.encoder_layer, num_encoder_layers)

        # Linear layer to produce the 1x48 output
        self.decoder = nn.Linear(d_model, 48)

    def forward(self, src):
        # Get the output from the transformer encoder
        encoder_output = self.transformer_encoder(src)

        # Use the last timestep of the encoder output for prediction
        output = self.decoder(encoder_output[-1])
        return output


In [100]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0

    for i, (src, tgt) in enumerate(loader):
        src = src.permute(1, 0, 2)
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = model(src)
        loss = criterion(output, tgt)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader)

In [101]:
# Hyperparameters
d_model = 48  # size of each timestep
nhead = 4  # number of heads in multihead attention
num_encoder_layers = 2  # number of encoder layers
dim_feedforward = 2048  # size of feedforward network in transformer
lr = 0.001  # learning rate

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Initialize model, criterion, and optimizer
model = Seq2PointTransformer(d_model, nhead, num_encoder_layers, dim_feedforward).to(device)
criterion = nn.MSELoss()  # Assuming regression task, modify if needed
optimizer = torch.optim.Adam(model.parameters(), lr=lr)

# Create dataset and dataloader
dataset = TimeSeriesDataset(df_pivot)
loader = torch.utils.data.DataLoader(dataset, batch_size=32, shuffle=True)


In [102]:

# Train the model
epochs = 10  # or any other number
for epoch in range(epochs):
    loss = train(model, loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss:.4f}")

Epoch 1/10, Loss: 0.3955
Epoch 2/10, Loss: 0.3944
Epoch 3/10, Loss: 0.3944
Epoch 4/10, Loss: 0.3944
Epoch 5/10, Loss: 0.3944
Epoch 6/10, Loss: 0.3944
Epoch 7/10, Loss: 0.3944
Epoch 8/10, Loss: 0.3944
Epoch 9/10, Loss: 0.3944
Epoch 10/10, Loss: 0.3944


In [ ]:
class TimeSeriesDataset(Dataset):
    def __init__(self, df_pivot):
        self.df = df_pivot

    def __len__(self):
        return len(self.df) * (self.df.columns.size - 4)  # sliding window of 4

    def __getitem__(self, idx):
        row_idx = idx // (self.df.columns.size - 4)
        col_idx = idx % (self.df.columns.size - 4)

        src_values = [self.df.iloc[row_idx, col_idx + i] for i in range(4)]
        x = torch.tensor(src_values, dtype=torch.float32)

        y_values = self.df.iloc[row_idx, col_idx + 4]
        y = torch.tensor(y_values, dtype=torch.float32)

        return x, y

In [103]:
len(df_pivot) * (df_pivot.columns.size - 4)

369600

In [114]:
df_pivot.columns.size - 4

11

In [119]:
idx=1
row_idx = idx // (df_pivot.columns.size - 4)
col_idx = idx % (df_pivot.columns.size - 4)



In [122]:
row_idx, col_idx

(0, 1)

In [120]:
src_values = [df_pivot.iloc[row_idx, col_idx + i] for i in range(4)]

In [121]:
src_values

[array([ 0.5743986 ,  0.7089401 ,  0.5678579 , -0.93600154,  0.1867537 ,
        -0.21889889,  0.31373662, -0.01483034,  0.58870435,  0.18307425,
         0.7581944 , -0.9867927 , -0.7873355 , -0.71986914,  0.26735574,
         0.5106258 ,  0.4649657 ,  0.8735933 ,  0.29587683,  0.59268314,
         0.72352916, -0.147296  ,  0.8954065 , -0.5896635 , -0.8418321 ,
         0.09008288,  0.32124034, -0.68763906, -0.00990593, -0.9052265 ,
         0.663454  , -0.742914  , -0.9030928 ,  0.86339515,  0.85623467,
         0.59408575,  0.9830612 , -0.86469734, -0.9600872 ,  0.74955744,
        -0.5013439 , -0.74300724, -0.28472558, -0.73516333,  0.6491805 ,
        -0.71408135,  0.58846635,  0.48585114], dtype=float32),
 array([-0.8775012 ,  0.77081597,  0.34914115,  0.9433937 ,  0.30284655,
        -0.23393598, -0.6151612 , -0.9200938 ,  0.26370117, -0.9513705 ,
         0.8736909 ,  0.8613019 ,  0.68549484,  0.49179533, -0.85623807,
        -0.60185134,  0.12894002, -0.77521306,  0.22900137, 

In [123]:
y_values = df_pivot.iloc[row_idx, col_idx + 4]
y_values

array([-0.3553231 , -0.37004066,  0.40704593, -0.8580441 , -0.6089973 ,
       -0.01253396,  0.79701287, -0.54673165,  0.2516961 , -0.9051518 ,
       -0.11013753,  0.2479235 , -0.94116247, -0.04404839, -0.24609083,
       -0.87675124,  0.15365578,  0.5742015 ,  0.97207564,  0.9036467 ,
        0.44921756,  0.86972296, -0.6813044 , -0.919421  , -0.28290847,
        0.67544866, -0.8999588 , -0.53313994, -0.39109695, -0.35104257,
        0.8720883 ,  0.7148386 ,  0.2416508 ,  0.9105341 , -0.6380717 ,
       -0.96881825, -0.27899995, -0.755492  ,  0.4419543 , -0.14857581,
       -0.5029816 ,  0.8295974 ,  0.07917184, -0.5547602 , -0.4101441 ,
        0.36330754,  0.99315625,  0.10865767], dtype=float32)

In [38]:
class SequenceDataset(Dataset):
    def __init__(self, df_pivot):
        self.data = []
        for coords, values in df_pivot.iterrows():
            self.data.append(values.values)
        self.data = np.stack(self.data)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        source = self.data[idx, :1199, :4]  # First 4 values of v for 1199 timesteps
        target = self.data[idx, 1:, 4:5]    # 5th value of v for 1199 timesteps
        return torch.tensor(source, dtype=torch.float32), torch.tensor(target, dtype=torch.float32)

In [39]:
dataset = SequenceDataset(df_pivot)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [40]:
import math

In [41]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return self.dropout(x)

class TransformerModel(nn.Module):
    def __init__(self, ntoken, d_model, nhead, nhid, nlayers, dropout=0.5):
        super(TransformerModel, self).__init__()

        # Embedding layers
        self.encoder_embedding = nn.Embedding(ntoken, d_model)
        self.decoder_embedding = nn.Embedding(ntoken, d_model)

        # Positional Encoders
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        self.pos_decoder = PositionalEncoding(d_model, dropout)

        # Transformer
        self.transformer = nn.Transformer(d_model, nhead, nlayers, nlayers, nhid, dropout)

        # Final Decoder
        self.fc_out = nn.Linear(d_model, ntoken)

        self.d_model = d_model
        self.ntoken = ntoken

        self.init_weights()

    def _generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def init_weights(self):
        initrange = 0.1
        self.encoder_embedding.weight.data.uniform_(-initrange, initrange)
        self.decoder_embedding.weight.data.uniform_(-initrange, initrange)
        self.fc_out.bias.data.zero_()
        self.fc_out.weight.data.uniform_(-initrange, initrange)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None, memory_mask=None, src_key_padding_mask=None, tgt_key_padding_mask=None, memory_key_padding_mask=None):
        # Embed and add positional encoding
        src = self.encoder_embedding(src) * math.sqrt(self.d_model)
        src = self.pos_encoder(src)

        tgt = self.decoder_embedding(tgt) * math.sqrt(self.d_model)
        tgt = self.pos_decoder(tgt)

        output = self.transformer(src, tgt, src_mask, tgt_mask, memory_mask, src_key_padding_mask, tgt_key_padding_mask, memory_key_padding_mask)
        output = self.fc_out(output)
        return output


In [42]:
ntokens = 48  # Assuming the dimension of 'v' is 48
d_model = 512  # Embedding dimension
nhead = 8     # Number of attention heads
nhid = 512    # Feed-forward hidden dimension
nlayers = 3   # Number of transformer layers
dropout = 0.5 # Dropout rate
epochs = 10   # Number of training epochs
lr = 0.001    # Learning rate

# Initialize model, criterion, and optimizer
model = TransformerModel(ntokens, d_model, nhead, nhid, nlayers, dropout)

In [43]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

num_epochs = 10
for epoch in range(num_epochs):
    for batch_idx, (src, target) in enumerate(dataloader):
        optimizer.zero_grad()
        output = model(src)
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()

        if batch_idx % 10 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Batch [{batch_idx+1}/{len(dataloader)}], Loss: {loss.item():.4f}")


IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed

In [1]:
import torch
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

In [2]:
# Assuming df_pivot is a pandas dataframe with the specified structure.
# Let's simulate a simple example dataframe since we cannot load actual data here.

# Example structure of df_pivot
# MultiIndex: ['x', 'y', 'z']
# Columns: [1, 2, ..., 1200]
# Each cell: [48-element vector]

# Simulating the dataframe structure
coordinates = [(i, j, k) for i in range(5) for j in range(5) for k in range(5)]
times = list(range(1, 121))  # Example with 120 time steps
index = pd.MultiIndex.from_tuples(coordinates, names=["x", "y", "z"])
df_pivot_example = pd.DataFrame(index=index, columns=times)

# Populate the dataframe with random vectors
for time in times:
    for coordinate in coordinates:
        df_pivot_example.at[coordinate, time] = np.random.rand(48)


In [3]:
df_pivot_example

1    \
x y z                                                      
0 0 0  [0.6878424390871023, 0.2768212783075963, 0.847...   
    1  [0.5779266301669413, 0.3731762595007664, 0.773...   
    2  [0.9447342528628159, 0.182913159885168, 0.2998...   
    3  [0.5372147252943944, 0.6171229193305061, 0.994...   
    4  [0.7501268331751132, 0.38472508899901037, 0.87...   
...                                                  ...   
4 4 0  [0.8095317890042081, 0.3191820142475976, 0.133...   
    1  [0.12798302960177332, 0.4956940136765082, 0.89...   
    2  [0.34974490612337605, 0.887306043438741, 0.209...   
    3  [0.16160053496950366, 0.3420349448611899, 0.51...   
    4  [0.09282386614559823, 0.624871445849829, 0.022...   

                                                     2    \
x y z                                                      
0 0 0  [0.5809579793190177, 0.4459698073835544, 0.367...   
    1  [0.23591294548615627, 0.15845101305507991, 0.3...   
    2  [0.3323447027102425, 0.1401148080621739, 0.898...   
    3  [0.5278870726356104, 0.29409766171847274, 0.34...   
    4  [0.5663890619166583, 0.8808382684326859, 0.449...   
...                                                  ...   
4 4 0  [0.15658274028013297, 0.30170296425863097, 0.8...   
    1  [0.0898750532311644, 0.6524062386325735, 0.322...   
    2  [0.7860431842606783, 0.3060448687903762, 0.847...   
    3  [0.09041452961470853, 0.5409152581493174, 0.08...   
    4  [0.5007969598953245, 0.6095920587835698, 0.086...   

                                                     3    \
x y z                                                      
0 0 0  [0.6313974797749081, 0.8291868394692316, 0.198...   
    1  [0.46976120907916086, 0.05760936936896932, 0.6...   
    2  [0.8247331792254158, 0.9324419540572725, 0.322...   
    3  [0.6111092538018336, 0.05166158161032053, 0.02...   
    4  [0.2837075730604809, 0.7930479348625339, 0.963...   
...                                                  ...   
4 4 0  [0.8761041408244822, 0.13498901205017322, 0.73...   
    1  [0.48413046033700713, 0.9545500581780986, 0.29...   
    2  [0.2422971727023383, 0.07815961143057526, 0.82...   
    3  [0.5092775480419792, 0.4456482006986413, 0.138...   
    4  [0.1211545830521421, 0.25697345992784393, 0.51...   

                                                     4    \
x y z                                                      
0 0 0  [0.9973311316410395, 0.9146604608614963, 0.326...   
    1  [0.8698291569750874, 0.20114163667834206, 0.84...   
    2  [0.31643119823824406, 0.5140978764068964, 0.28...   
    3  [0.9194081177611777, 0.19053155398254018, 0.82...   
    4  [0.9100545455038117, 0.2900948829248089, 0.514...   
...                                                  ...   
4 4 0  [0.8148075324987097, 0.6578619672897413, 0.436...   
    1  [0.853335431549608, 0.8003205139546236, 0.3389...   
    2  [0.9779636791138482, 0.3469181178183982, 0.075...   
    3  [0.50251603605952, 0.9749874784069785, 0.58873...   
    4  [0.38370901344146735, 0.6743249281914474, 0.58...   

                                                     5    \
x y z                                                      
0 0 0  [0.45538357267505447, 0.9819753466479034, 0.01...   
    1  [0.7306318860111701, 0.6825347292830675, 0.013...   
    2  [0.4851945469298825, 0.10303790235610988, 0.90...   
    3  [0.5661328735535939, 0.10409315864806556, 0.15...   
    4  [0.8130646852451312, 0.6313139250294766, 0.798...   
...                                                  ...   
4 4 0  [0.08261349307573407, 0.8216915275913836, 0.45...   
    1  [0.9724157934428228, 0.1961983663926924, 0.574...   
    2  [0.07936999384484023, 0.8911176743395798, 0.83...   
    3  [0.9060764929595394, 0.8798973767333527, 0.997...   
    4  [0.22221496199869595, 0.9993418125019373, 0.18...   

                                                     6    \
x y z                                                      
0 0 0  [0.3348382878546371, 

In [6]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, sequence_length):
        self.dataframe = dataframe
        self.sequence_length = sequence_length

        # Flatten the multi-index and reset it to prepare for use in PyTorch
        self.flat_dataframe = self.dataframe.reset_index()

    def __len__(self):
        # The length of the dataset is the number of sequences that can be created
        # from the dataframe given the sequence length
        return (len(self.dataframe.columns) - self.sequence_length) * len(self.dataframe)

    def __getitem__(self, idx):
        # Calculate row and time index from single index
        row_idx = idx // (len(self.dataframe.columns) - self.sequence_length)
        time_idx = idx % (len(self.dataframe.columns) - self.sequence_length)

        # Select the current location
        current_location = self.flat_dataframe.iloc[row_idx, :3].values
        current_location = np.repeat(current_location[np.newaxis, :], self.sequence_length, axis=0)

        # Select a sequence of time steps for the given index
        time_seq = self.dataframe.iloc[row_idx, time_idx:time_idx+self.sequence_length+1].values

        # Prepare the source and target sequences
        # The source sequence is the current sequence of velocity vectors
        src = np.array([np.fromstring(v[1:-1], sep=' ') for v in time_seq[:-1]])
        # The target sequence is the sequence shifted by one time step
        tgt = np.array([np.fromstring(v[1:-1], sep=' ') for v in time_seq[1:]])

        # Concatenate the spatial coordinates to the source sequence
        src = np.hstack((current_location, src))

        # Convert to tensors
        src = torch.tensor(src, dtype=torch.float32)
        tgt = torch.tensor(tgt, dtype=torch.float32)

        return src, tgt

In [7]:
# Instantiate the dataset
sequence_length = 5  # For example, we use 5 time steps to predict the next step
dataset = SpatioTemporalDataset(df_pivot_example, sequence_length)

# Create a DataLoader
batch_size = 4
dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# Example of how to get a batch from the dataloader
for src, tgt in dataloader:
    print("Source batch shape:", src.shape)
    print("Target batch shape:", tgt.shape)
    break  # We break the loop after one batch for this example

# Output the shapes of the source and target batches
print("Source batch example:", src[0])
print("Target batch example:", tgt[0])

/tmp/ipykernel_151402/474041406.py:28: DeprecationWarning: string or file could not be read to its end due to unmatched data; this will raise a ValueError in the future.
  src = np.array([np.fromstring(v[1:-1], sep=' ') for v in time_seq[:-1]])


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (5,) + inhomogeneous part.

In [9]:
all_dfs = []

for i in range(1, 16):
    df = pd.read_pickle(f"/mnt/d/sources/cgan/standalone/dataset/latent_representation_for_{i}.pkl.zip", compression="zip")
    step = len(df) // 3360
    sampled_df = df.iloc[::step].copy()
    sampled_df = sampled_df[['x', 'y', 'z', 'time', 'latent_representation']]
    sampled_df['latent_representation'] = sampled_df['latent_representation'].apply(lambda x: x[0])
    all_dfs.append(sampled_df)

df_combined = pd.concat(all_dfs, ignore_index=True)
df_pivot = df_combined.pivot_table(index=['x', 'y', 'z'], columns='time', values='latent_representation',
                                   aggfunc='first')

In [10]:
df_pivot.head()

time                                                         1   \
x    y   z                                                        
-117 -76 -25  [0.60152996, 0.98103255, -0.008870157, -0.2454...   
         -21  [0.096365936, 0.54300946, 0.8387079, 0.4684101...   
         -17  [0.6943935, 0.6303967, -0.8195155, 0.9442892, ...   
         -13  [0.56154877, 0.7658759, -0.31268513, 0.9745437...   
         -9   [0.9743955, -0.93269074, 0.027911449, 0.220510...   

time                                                         2   \
x    y   z                                                        
-117 -76 -25  [0.5743986, 0.7089401, 0.5678579, -0.93600154,...   
         -21  [-0.79866904, -0.12209187, 0.91154915, 0.76043...   
         -17  [0.9876632, -0.61406004, -0.3469003, 0.4697731...   
         -13  [-0.7751982, -0.325407, 0.4743654, -0.13019444...   
         -9   [-0.3710597, -0.19291241, 0.8153351, -0.286354...   

time                                                         3   \
x    y   z                                                        
-117 -76 -25  [-0.8775012, 0.77081597, 0.34914115, 0.9433937...   
         -21  [0.22371271, -0.7035653, -0.24191181, -0.04110...   
         -17  [-0.94120944, 0.9091235, 0.63393474, 0.7522126...   
         -13  [0.31826624, 0.65888816, 0.98721373, -0.370199...   
         -9   [-0.9283217, 0.25671372, 0.90663964, 0.91436, ...   

time                                                         4   \
x    y   z                                                        
-117 -76 -25  [-0.21666437, 0.276739, 0.67670155, -0.8597325...   
         -21  [-0.12166012, -0.90468264, -0.703625, -0.35087...   
         -17  [0.9533979, 0.374611, 0.08358036, 0.45856717, ...   
         -13  [-0.73936015, 0.13778922, -0.73885727, -0.4434...   
         -9   [-0.44492516, -0.2988802, -0.15240346, -0.3305...   

time                                                         5   \
x    y   z                                                        
-117 -76 -25  [0.36412022, -0.76997614, 0.266663, -0.4527933...   
         -21  [-0.6756259, 0.20125556, 0.41114074, -0.192610...   
         -17  [0.103173874, -0.9596432, -0.21567442, 0.48207...   
         -13  [0.95708394, 0.9988378, -0.84009784, -0.483040...   
         -9   [-0.5891063, -0.2228696, -0.18154225, -0.36551...   

time                                                         6   \
x    y   z                                                        
-117 -76 -25  [-0.3553231, -0.37004066, 0.40704593, -0.85804...   
         -21  [-0.2619261, -0.9286362, 0.87315434, 0.1708212...   
         -17  [0.8844934, -0.4611117, -0.35537854, -0.904041...   
         -13  [-0.43986663, -0.8859522, -0.60281193, -0.9730...   
         -9   [0.70254093, 0.91327983, 0.54900235, 0.8643656...   

time                                                         7   \
x    y   z                                                        
-117 -76 -25  [0.4651682, 0.32191125, -0.8944155, 0.7075796,...   
         -21  [0.12846749, -0.70874083, -0.09848327, 0.54393...   
         -17  [0.19190139, -0.2850105, 0.09330799, 0.2111620...   
         -13  [0.628395, 0.42571187, -0.22475213, -0.5347901...   
         -9   [-0.8378765, -0.8759353, 0.020267947, -0.99677...   

time                                                         8   \
x    y   z                                                        
-117 -76 -25  [-0.7426088, 0.7396835, -0.4338635, 0.6320076,...   
         -21  [-0.9192027, 0.39118025, 0.6679161, -0.7074012...   
         -17  [-0.55416745, 0.42543814, 0.32234558, -0.66634...   
         -13  [0.8604549, -0.80334365, -0.19985121, 0.601780...   
         -9   [-0.43921062, 0.9817179, 0.47819474, 0.8692322...   

time                                                         9   \
x    y   z                                                        
-117 -76 -25  [-0.16479653, 0.6672011, 0.44587672, -0.632495...   
         -21  [0.75533044, -0.95989746,

In [11]:
df_pivot_flat = df_pivot.reset_index()

In [12]:
df_pivot_flat.head()

time,x,y,z,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15
0,-117,-76,-25,"[0.60152996, 0.98103255, -0.008870157, -0.2454...","[0.5743986, 0.7089401, 0.5678579, -0.93600154,...","[-0.8775012, 0.77081597, 0.34914115, 0.9433937...","[-0.21666437, 0.276739, 0.67670155, -0.8597325...","[0.36412022, -0.76997614, 0.266663, -0.4527933...","[-0.3553231, -0.37004066, 0.40704593, -0.85804...","[0.4651682, 0.32191125, -0.8944155, 0.7075796,...","[-0.7426088, 0.7396835, -0.4338635, 0.6320076,...","[-0.16479653, 0.6672011, 0.44587672, -0.632495...","[-0.82881856, -0.4417005, -0.93003106, -0.8619...","[0.73687744, -0.4033748, -0.95862985, -0.47497...","[0.04219211, 0.5056695, -0.7162363, -0.4815756...","[-0.25694436, 0.4484407, -0.7467118, -0.507902...","[0.37512484, -0.87216157, -0.27912727, -0.6371...","[-0.8197715, -0.40555948, 0.9871446, -0.213071..."
1,-117,-76,-21,"[0.096365936, 0.54300946, 0.8387079, 0.4684101...","[-0.79866904, -0.12209187, 0.91154915, 0.76043...","[0.22371271, -0.7035653, -0.24191181, -0.04110...","[-0.12166012, -0.90468264, -0.703625, -0.35087...","[-0.6756259, 0.20125556, 0.41114074, -0.192610...","[-0.2619261, -0.9286362, 0.87315434, 0.1708212...","[0.12846749, -0.70874083, -0.09848327, 0.54393...","[-0.9192027, 0.39118025, 0.6679161, -0.7074012...","[0.75533044, -0.95989746, -0.90374166, -0.0630...","[0.23623419, 0.62348735, 0.38812354, 0.3865175...","[-0.7789519, 0.37796363, -0.87555295, -0.03135...","[0.7806267, 0.8975115, -0.5607745, -0.9473221,...","[0.95479465, 0.7803736, 0.1497075, -0.9292824,...","[-0.024801627, -0.3578735, -0.9100004, -0.1973...","[0.62667376, 0.6798731, 0.34012732, 0.5542102,..."
2,-117,-76,-17,"[0.6943935, 0.6303967, -0.8195155, 0.9442892, ...","[0.9876632, -0.61406004, -0.3469003, 0.4697731...","[-0.94120944, 0.9091235, 0.63393474, 0.7522126...","[0.9533979, 0.374611, 0.08358036, 0.45856717, ...","[0.103173874, -0.9596432, -0.21567442, 0.48207...","[0.8844934, -0.4611117, -0.35537854, -0.904041...","[0.19190139, -0.2850105, 0.09330799, 0.2111620...","[-0.55416745, 0.42543814, 0.32234558, -0.66634...","[0.6108569, 0.79253775, -0.007248677, 0.986103...","[-0.060510974, 0.8598963, -0.14712083, 0.79679...","[-0.16263914, 0.09215525, -0.3731518, 0.634308...","[-0.26758555, -0.94273233, 0.45466158, 0.84968...","[-0.60795283, -0.4063541, 0.5913332, -0.144027...","[-0.7996373, 0.7963824, 0.7067519, -0.98572326...","[0.8420596, 0.69856083, 0.15949044, 0.58392566..."
3,-117,-76,-13,"[0.56154877, 0.7658759, -0.31268513, 0.9745437...","[-0.7751982, -0.325407, 0.4743654, -0.13019444...","[0.31826624, 0.65888816, 0.98721373, -0.370199...","[-0.73936015, 0.13778922, -0.73885727, -0.4434...","[0.95708394, 0.9988378, -0.84009784, -0.483040...","[-0.43986663, -0.8859522, -0.60281193, -0.9730...","[0.628395, 0.42571187, -0.22475213, -0.5347901...","[0.8604549, -0.80334365, -0.19985121, 0.601780...","[0.62275165, 0.250471, -0.2056058, 0.48035592,...","[0.9358812, -0.9361252, -0.4325922, 0.56805587...","[0.5729953, 0.9068877, 0.5152233, 0.42906854, ...","[-0.7450801, 0.84187305, 0.71652573, -0.918943...","[-0.41168693, -0.66889644, 0.18859309, -0.2000...","[0.007128273, 0.85873616, 0.58662, -0.6822411,...","[0.17318805, -0.8831455, 0.7376071, 0.01548677..."
4,-117,-76,-9,"[0.9743955, -0.93269074, 0.027911449, 0.220510...","[-0.3710597, -0.19291241, 0.8153351, -0.286354...","[-0.9283217, 0.25671372, 0.90663964, 0.91436, ...","[-0.44492516, -0.2988802, -0.15240346, -0.3305...","[-0.5891063, -0.2228696, -0.18154225, -0.36551...","[0.70254093, 0.91327983, 0.54900235, 0.8643656...","[-0.8378765, -0.8759353, 0.020267947, -0.99677...","[-0.43921062, 0.9817179, 0.47819474, 0.8692322...","[0.975531, 0.67379576, 0.41823196, 0.63230884,...","[-0.6277772, 0.05303601, 0.62489265, -0.050239...","[0.97310853, -0.43587416, -0.16551034, -0.4941...","[0.66972435, 0.10993393, -0.88734424, 0.509057...","[0.17666648, 0.3667334, -0.90357393, -0.942890...","[-0.15619136, 0.62931603, -0.69244576, -0.7722...","[0.94355583, -0.54230475, 0.96

In [13]:
df_pivot_flat.columns

Index(['x', 'y', 'z', 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15], dtype='object', name='time')

In [15]:
df_pivot_flat.dtypes

time
x      int64
y      int64
z      int64
1     object
2     object
3     object
4     object
5     object
6     object
7     object
8     object
9     object
10    object
11    object
12    object
13    object
14    object
15    object
dtype: object

In [40]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length

    def __len__(self):
        return len(self.dataframe) - self.sequence_length

    def __getitem__(self, idx):
        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

        # Prepare the sequence of latent representations
        sequences = []
        for t in range(self.sequence_length + 1):  # plus one for the target
            time_col = str(idx + t + 1)  # assuming time columns are labeled as strings "1", "2", ...
            vector = self.dataframe[int(time_col)].iloc[idx]
            sequences.append(vector)

        # Convert the list of arrays into a 2D array
        sequences = np.stack(sequences)

        # Normalize time and spatial coordinates if necessary
        # ...

        # Split the sequences into source and target
        src_seq = sequences[:-1]  # All but the last
        tgt_seq = sequences[-1]  # Only the last

        # Combine the spatial coordinates with the source sequence
        src_seq = np.concatenate([np.tile(coordinates, (self.sequence_length, 1)), src_seq], axis=1)

        return torch.tensor(src_seq, dtype=torch.float), torch.tensor(tgt_seq, dtype=torch.float)

In [145]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length
        # self.num_samples = len(self.dataframe) - self.sequence_length
        self.num_samples = len(self.dataframe) - self.sequence_length + 1
        self.max_time_frame = self.dataframe.columns[-1]
        print(self.max_time_frame)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        if idx >= self.num_samples:
            raise IndexError("Index out of range")

        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

        # Prepare the sequence of latent representations
        sequences = []
        for t in range(self.sequence_length + 1):  # plus one for the target
            time_col = idx + t + 1
            if time_col > self.max_time_frame:
                raise IndexError(f"Time column {time_col} not found in the dataframe")
            if int(time_col) not in self.dataframe.columns:
                raise IndexError(f"Time column {time_col} not found in the dataframe")

            vector = self.dataframe[int(time_col)].iloc[idx]
            sequences.append(vector)

        # Convert the list of arrays into a 2D array
        sequences = np.stack(sequences)

        # Split the sequences into source and target
        src_seq = sequences[:-1]  # All but the last
        tgt_seq = sequences[-1]  # Only the last

        # Combine the spatial coordinates with the source sequence
        src_seq = np.concatenate([np.tile(coordinates, (self.sequence_length, 1)), src_seq], axis=1)

        return torch.tensor(src_seq, dtype=torch.float), torch.tensor(tgt_seq, dtype=torch.float)


In [141]:
class SpatioTemporalDataset(Dataset):
    def __init__(self, dataframe, sequence_length=5):
        self.dataframe = dataframe
        self.sequence_length = sequence_length
        # self.num_samples = len(self.dataframe) - self.sequence_length
        self.num_samples = len(self.dataframe) - self.sequence_length + 1
        self.max_time_frame = self.dataframe.columns[-1]
        print(self.max_time_frame)

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        # Retrieve spatial coordinates
        coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)
    
        # Prepare the sequence of latent representations
        sequences = []
        max_time_col = self.dataframe.columns[-1]  # Assuming the last column is the last time frame
        start_time_col = max(idx + 1, 1)  # Ensure starting at least from the first time frame
        end_time_col = min(start_time_col + self.sequence_length, int(max_time_col))
    
        for time_step in range(start_time_col, end_time_col + 1):
            time_col = str(time_step)  # Convert time step to string to match column names
            vector = self.dataframe[time_col].iloc[idx]
            sequences.append(vector)
    
        # Handle cases where there aren't enough future time steps
        while len(sequences) < self.sequence_length + 1:
            sequences.append(np.zeros_like(vector))
    
        # Convert the list of arrays into a 2D array
        sequences = np.stack(sequences)
    
        # Normalize time and spatial coordinates if necessary
        # ...
    
        # Split the sequences into source and target
        src_seq = sequences[:-1]  # All but the last
        tgt_seq = sequences[-1]  # Only the last
    
        # Combine the spatial coordinates with the source sequence
        src_seq = np.concatenate([np.tile(coordinates, (self.sequence_length, 1)), src_seq], axis=1)
    
        return torch.tensor(src_seq, dtype=torch.float), torch.tensor(tgt_seq, dtype=torch.float)



In [105]:
for t in range(4 + 1):  # plus one for the target
    print(t)
    time_col = str(0 + t + 1)  # assuming time columns are labeled as strings "1", "2", ...
    # print(time_col)
    vector = df_pivot_flat[int(time_col)]
    # vector = self.dataframe[time_col].iloc[idx]

0
1
2
3
4


In [146]:
dataset = SpatioTemporalDataset(df_pivot_flat, 4)

15


In [151]:
dataset.__getitem__(15)

IndexError: Time column 16 not found in the dataframe

In [152]:
dataloader = DataLoader(dataset, batch_size=5, shuffle=False, drop_last=True)

In [153]:
i=0
for src_seq, tgt_seq in dataloader:
    print(src_seq)
    print(src_seq.shape)
    print(tgt_seq)
    print(tgt_seq.shape)
    if i==1:
        break
    i+=1

tensor([[[-1.1700e+02, -7.6000e+01, -2.5000e+01,  ..., -2.3544e-01,
           3.6328e-01, -1.1773e-01],
         [-1.1700e+02, -7.6000e+01, -2.5000e+01,  ..., -7.1408e-01,
           5.8847e-01,  4.8585e-01],
         [-1.1700e+02, -7.6000e+01, -2.5000e+01,  ...,  9.4597e-01,
          -8.6952e-03,  9.7801e-03],
         [-1.1700e+02, -7.6000e+01, -2.5000e+01,  ...,  3.1899e-01,
           9.0584e-01,  5.2840e-01]],

        [[-1.1700e+02, -7.6000e+01, -2.1000e+01,  ..., -6.9382e-01,
          -3.4793e-01,  3.7007e-01],
         [-1.1700e+02, -7.6000e+01, -2.1000e+01,  ..., -5.2397e-01,
           6.2920e-01,  8.4639e-01],
         [-1.1700e+02, -7.6000e+01, -2.1000e+01,  ...,  3.0185e-01,
          -3.2697e-01,  4.3634e-01],
         [-1.1700e+02, -7.6000e+01, -2.1000e+01,  ..., -5.2382e-01,
           9.6679e-01,  2.0955e-02]],

        [[-1.1700e+02, -7.6000e+01, -1.7000e+01,  ..., -7.9497e-01,
           9.8518e-01,  3.5504e-01],
         [-1.1700e+02, -7.6000e+01, -1.7000e+01,  .

In [ ]:
def __getitem__(self, idx):
    # Retrieve spatial coordinates
    coordinates = self.dataframe.iloc[idx][['x', 'y', 'z']].values.astype(np.float32)

    # Prepare the source sequence for the given window
    src_seq = []
    for t in range(idx, idx + self.sequence_length):  # Get the source sequence
        time_cols = self.dataframe.columns[3+t:3+t+1]  # Time columns start after x, y, z
        vectors = self.dataframe.iloc[idx][time_cols].values
        src_seq.append(vectors)

    # The target is the latent representation at the next time step
    tgt_time_col = self.dataframe.columns[3+idx+self.sequence_length]
    tgt_seq = self.dataframe.iloc[idx][tgt_time_col]

    src_seq = np.stack(src_seq)

    # Combine the spatial coordinates with the source sequence
    src_seq_with_coordinates = np.concatenate([np.tile(coordinates, (self.sequence_length, 1)), src_seq], axis=1)

    return torch.tensor(src_seq_with_coordinates, dtype=torch.float), torch.tensor(tgt_seq, dtype=torch.float)
